## Gold Baseline Modeling (Deliverable 1.3.2)

This notebook implements the baseline anomaly detection model using a single, broad Isolation Forest trained on the complete Gold feature set. The baseline serves as the primary comparison point for evaluating whether the three-stage cascade model improves alert quality.

**Purpose:**  
To produce the baseline model’s anomaly scores and alert outputs using the fully preprocessed Gold dataset. These outputs represent the simplest form of unsupervised anomaly detection and form the quantitative reference for the comparative evaluation in the Gold Comparison notebook.

**Key Goals:**

- Load the Gold preprocessed dataset and Stage 1 feature set.
- Train a single Isolation Forest using all vetted numeric features.
- Generate anomaly scores and binary alert flags.
- Produce baseline alert frequency counts, false-positive counts, and normal-period alert rates.
- Save all baseline model artifacts, metrics, and alert outputs for use in the Gold Comparison notebook.

**Relevance to Section C:**  
This notebook provides the baseline alert patterns used in Section C to evaluate whether the cascade reduces false positives, reduces noisy alerts, and preserves meaningful anomaly sensitivity. The outputs here form the “reference condition” for the paired statistical tests and practical significance analysis.

## Overview

This notebook trains and evaluates the broad baseline Isolation Forest model for comparison against cascade models. It is part of the Gold portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook trains and evaluates the broad baseline Isolation Forest model for comparison against cascade models.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Connect the modeling or validation outputs back to the final anomaly-detection evidence used in the capstone write-up.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/EDA_Notebook_Pump_Gold_02_Baseline_Modeling_code_reference.md`


## Baseline Modeling Setup and Imports

In this section I am loading the libraries and project utilities needed for the Gold baseline modeling stage.

The purpose here is to get the notebook ready before I touch the Gold modeling data. That includes:
- standard Python libraries
- model and metrics utilities
- project path and config loading
- logging
- truth-record utilities
- artifact saving helpers
- experiment tracking support

At this point, I am not fitting the model yet. I am just setting up the notebook so the baseline workflow can run in a structured and repeatable way.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union, Mapping, cast

import math
from pathlib import Path
import yaml
import os
import logging
import wandb

import pandas as pd 
import numpy as np 

import joblib 

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import (
    StandardScaler, 
    MinMaxScaler, 
    OneHotEncoder, 
    RobustScaler,
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import (
    RandomForestClassifier, 
    IsolationForest,
)

from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    precision_recall_fscore_support, 
    roc_auc_score, 
    average_precision_score,
)
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor


# Custom Utilities Module
from utils.core.paths import get_paths
from utils.core.file_io import (
    load_data, 
    save_data, 
    save_json, 
    load_json,
)

from utils.core.logging_profiler import profile_dataframe
from utils.core.logging_setup import (
    configure_logging, 
    log_layer_paths,
)

from utils.core.wandb_utils import finalize_wandb_stage

from utils.core.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.core.config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.medallion.gold.cascade_row_tracking import (
    ensure_stable_row_id,
    build_stage_scoring_frame,
    score_isolation_forest_stage,
    merge_stage_results_back,
    finalize_stage_flag_columns,
    get_detected_rows_dataframe,
    get_stage_detected_rows_dataframe,
)

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)
from utils.database.layer_postgres import (
    read_layer_dataframe, 
    write_layer_dataframe, 
    prepare_layer_dataframe,
)

from utils.database.sql_notebook_helpers import (
    delete_dataset_run_rows,
    execute_many,
    get_existing_dataframe,
    get_row_value,
    log_data_quality_event,
    log_pipeline_artifact,
    preview_sql_table,
    row_to_payload,
    sql_table_ref,
    to_builtin,
    to_json_string,
    to_scalar,
    upsert_pipeline_run,
)

from utils.database.medallion_sql_writers import (
    log_gold_05_anomaly_detection_summary_sql,
    log_silver_eda_sql,
    write_bronze_sensor_observations_sql,
    write_gold_baseline_scores_sql,
    write_gold_cascade_scores_sql,
    write_gold_model_comparison_results_sql,
    write_gold_preprocessed_features_sql,
    write_silver_sensor_observation_features_sql,
)

# Ledger 
from utils.core.ledger import Ledger

# Artifacts
from utils.core.artifacts import (
    build_artifact_dirs_from_config,
    artifact_file_path,
)

from utils.core.notebook_context import load_notebook_context

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


----

In [2]:
def cfg_require_mapping(value: object, name: str) -> Mapping[str, Any]:
    if not isinstance(value, Mapping):
        raise TypeError(
            f"{name} must be a mapping, got {type(value).__name__}: {value!r}"
        )

    return cast(Mapping[str, Any], value)


def cfg_optional_mapping(value: object | None, name: str) -> Mapping[str, Any]:
    if value is None:
        return {}

    return cfg_require_mapping(value, name)

---

## Load Configuration, Paths, and Baseline Runtime Settings

Here I load the resolved configuration values that control how the baseline notebook runs.

This step defines the key runtime pieces for the baseline stage, such as:
- dataset identity
- Gold version and recipe information
- threshold settings
- estimator count
- train fraction
- file names and artifact paths
- truth-store locations
- model output locations

I like doing this early because it keeps the rest of the notebook cleaner. Instead of scattering settings throughout the notebook, I define them once here and reuse them in the later steps.

In [3]:


# Shared notebook context
CONTEXT_STAGE = "gold_baseline"
CONTEXT_DATASET = "pump"
CONTEXT_LAYER = "gold"
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"
CONTEXT_LOG_FILE = "gold_modeling_baseline.log"

CTX = load_notebook_context(
    stage=CONTEXT_STAGE,
    dataset=CONTEXT_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    logger_child_name="capstone.gold.baseline",
    log_filename=CONTEXT_LOG_FILE,
)

# Shared aliases used throughout the notebook
paths = CTX.paths
CONFIG = CTX.config
CONFIG_MAP = CTX.config
STAGE_CFG = CTX.stage_config
GOLD_CFG = CTX.stage_config
RESOLVED_PATHS = CTX.resolved_paths
FILENAMES = CTX.filenames
VERSIONS_CFG = CTX.versions
RUNTIME_CFG = CTX.runtime
DATASET_CFG = CTX.dataset_config
WANDB_CFG = CTX.wandb
EXECUTION_CFG = CTX.execution
PIPELINE = CTX.pipeline
logger = CTX.logger
ledger = CTX.ledger
LOG_PATH = CTX.log_path
CONTEXT_RECIPE_ID = CTX.recipe_id

logger.info(
    "Notebook context loaded",
    extra={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
    },
)

ledger.add(
    kind="step",
    step="context_loaded",
    message="Loaded shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-15 00:57:15,725 | INFO | capstone.gold.baseline | gold_baseline stage starting
2026-06-15 00:57:15,728 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T00:57:15.728007+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger from shared notebook context', 'why': None, 'consequence': None, 'data': {'stage': 'gold_baseline', 'recipe_id': 'gold_modeling__v001_baseline', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/logs/gold_modeling_baseline.log'}}
2026-06-15 00:57:15,730 | INFO | capstone.gold.baseline | Notebook context loaded
2026-06-15 00:57:15,732 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T00:57:15.732890+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'context_loaded', 'message': 'Loaded shared notebook context.', 'why': None, 'consequence': None, 'data': {'stage': 'g

{'ts_utc': '2026-06-15T00:57:15.732890+00:00',
 'stage': 'gold_baseline',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'context_loaded',
 'message': 'Loaded shared notebook context.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'gold_baseline',
  'recipe_id': 'gold_modeling__v001_baseline',
  'dataset': 'pump',
  'mode': 'train',
  'profile': 'default',
  'log_path': '/workspace/logs/gold_modeling_baseline.log'}}

In [4]:


TRUTH_CONFIG = cast(Dict[str, Any], build_truth_config_block(CONFIG))
TRUTH_CONFIG["pipeline"] = PIPELINE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Stage details ----

STAGE = "gold"

LAYER_NAME = str(GOLD_CFG["layer_name"])

RECIPE_ID = str(GOLD_CFG["recipe_id"])

GOLD_VERSION = str(VERSIONS_CFG["gold"])
TRUTH_VERSION = str(VERSIONS_CFG["truth"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

PIPELINE_MODE = str(PIPELINE["execution_mode"])
RUN_MODE = str(RUNTIME_CFG.get("mode", CONFIG_RUN_MODE))
CONFIG_PROFILE = str(RUNTIME_CFG.get("profile", CONFIG_PROFILE))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

DATASET_NAME_CONFIG = str(DATASET_CFG.get("name", "pump"))
DATASET_NAME = DATASET_NAME_CONFIG.strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

GOLD_PROCESS_RUN_ID = make_process_run_id(
    str(GOLD_CFG["process_run_id_prefix"])
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- W&B ----

WANDB_PROJECT = str(WANDB_CFG.get("project", "capstone"))
WANDB_ENTITY = str(WANDB_CFG.get("entity", ""))
WANDB_RUN_NAME = f"{GOLD_VERSION}"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- File names ----

GOLD_PREPROCESSED_FILE_NAME = str(FILENAMES["gold_preprocessed_file_name"])
GOLD_PREPROCESSED_SCALED_FILE_NAME = str(
    FILENAMES["gold_preprocessed_scaled_file_name"]
)

GOLD_FIT_FILE_NAME = str(FILENAMES["gold_fit_file_name"])
GOLD_TEST_FILE_NAME = str(FILENAMES["gold_test_file_name"])
GOLD_TRAIN_FILE_NAME = str(FILENAMES["gold_train_file_name"])

STAGE1_FEATURES_FILE_NAME = str(FILENAMES["stage1_features_file_name"])

BASELINE_RESULTS_FILE_NAME_CSV = str(FILENAMES["baseline_results_file_name_csv"])
BASELINE_RESULTS_FILE_NAME_PICKLE = str(
    FILENAMES["baseline_results_file_name_pickle"]
)

BASELINE_MODEL_FILE_NAME = str(FILENAMES["baseline_model_file_name"])
BASELINE_THRESHOLDS_FILE_NAME = str(FILENAMES["baseline_thresholds_file_name"])
BASELINE_SUMMARY_FILE_NAME = str(FILENAMES["baseline_summary_file_name"])
BASELINE_METADATA_FILE_NAME = str(FILENAMES["baseline_metadata_file_name"])

GOLD_BASELINE_LEDGER_FILE_NAME = str(
    FILENAMES.get("gold_baseline_ledger_file_name", "gold_baseline_ledger.jsonl")
)

TRUTH_INDEX_FILE_NAME = "truth_index.jsonl"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Runtime knobs ----

TRAIN_FRACTION = float(GOLD_CFG["train_fraction"])
RANDOM_SEED = int(GOLD_CFG["random_seed"])

BASELINE_THRESHOLD_PERCENTILE = float(GOLD_CFG["baseline_threshold_percentile"])
STAGE1_THRESHOLD_PERCENTILE = float(GOLD_CFG["stage1_threshold_percentile"])
STAGE2_THRESHOLD_PERCENTILE = float(GOLD_CFG["stage2_threshold_percentile"])

BASELINE_ESTIMATOR_COUNT = int(GOLD_CFG["baseline_estimator_count"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Base paths only ----

ARTIFACTS_ROOT = Path(str(RESOLVED_PATHS["artifacts_root"]))

GOLD_PREPROCESSED_DATA_PATH = Path(
    str(RESOLVED_PATHS["gold_preprocessed_data_path"])
)
GOLD_PREPROCESSED_SCALED_DATA_PATH = Path(
    str(RESOLVED_PATHS["gold_preprocessed_scaled_data_path"])
)

GOLD_TRAIN_DATA_PATH = Path(str(RESOLVED_PATHS["gold_train_data_path"]))
GOLD_TEST_DATA_PATH = Path(str(RESOLVED_PATHS["gold_test_data_path"]))
GOLD_FIT_DATA_PATH = Path(str(RESOLVED_PATHS["gold_fit_data_path"]))

GOLD_ARTIFACTS_PATH = Path(str(RESOLVED_PATHS["gold_artifacts_dir"]))

MODELS_PATH = Path(str(RESOLVED_PATHS["models_root"]))
BASELINE_MODELS_PATH = Path(str(RESOLVED_PATHS["baseline_models_path"]))

print(BASELINE_MODELS_PATH)

BASELINE_MODEL_ARTIFACT_PATH = Path(
    str(RESOLVED_PATHS["baseline_model_artifact_path"])
)

BASELINE_RESULTS_PATH_CSV = Path(str(RESOLVED_PATHS["baseline_results_path_csv"]))
BASELINE_RESULTS_PATH_PICKLE = Path(
    str(RESOLVED_PATHS["baseline_results_path_pickle"])
)

BASELINE_THRESHOLDS_PATH = Path(str(RESOLVED_PATHS["baseline_thresholds_path"]))
BASELINE_SUMMARY_PATH = Path(str(RESOLVED_PATHS["baseline_summary_path"]))
BASELINE_METADATA_PATH = Path(str(RESOLVED_PATHS["baseline_metadata_path"]))

TRUTHS_PATH = Path(str(RESOLVED_PATHS["truths_dir"]))
TRUTH_INDEX_PATH = Path(str(RESOLVED_PATHS["truth_index_path"]))

STAGE1_FEATURES_PATH = Path(str(RESOLVED_PATHS["stage1_features_path"]))

LOGS_PATH = Path(str(RESOLVED_PATHS["logs_root"]))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# W&B

set_wandb_dir_from_config(CONFIG)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# Path failsafes

GOLD_PREPROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_PREPROCESSED_SCALED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

GOLD_FIT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TEST_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TRAIN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

MODELS_PATH.mkdir(parents=True, exist_ok=True)
#BASELINE_MODELS_PATH.mkdir(parents=True, exist_ok=True)
BASELINE_MODELS_PATH.parent.mkdir(parents=True, exist_ok=True)
#BASELINE_MODEL_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

BASELINE_RESULTS_PATH_CSV.parent.mkdir(parents=True, exist_ok=True)
BASELINE_RESULTS_PATH_PICKLE.parent.mkdir(parents=True, exist_ok=True)
BASELINE_THRESHOLDS_PATH.parent.mkdir(parents=True, exist_ok=True)
BASELINE_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
BASELINE_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)

TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

print("Project root:", paths.root)
print("Artifacts root:", ARTIFACTS_ROOT)
print("CONFIG dataset:", DATASET_NAME_CONFIG)
print("Gold preprocessed path:", GOLD_PREPROCESSED_DATA_PATH)
print("Gold fit path:", GOLD_FIT_DATA_PATH)
print("Baseline model path:", BASELINE_MODEL_ARTIFACT_PATH)
print("Baseline results path:", BASELINE_RESULTS_PATH_CSV)

/workspace/models/pump/pump__gold__baseline_isolation_forest.joblib
Project root: /workspace
Artifacts root: /workspace/artifacts
CONFIG dataset: pump
Gold preprocessed path: /workspace/data/gold/pump__gold__preprocessed.parquet
Gold fit path: /workspace/data/gold/pump__gold__fit_normal_only.parquet
Baseline model path: /workspace/artifacts/gold/pump/baseline/models/pump__gold__baseline_isolation_forest.joblib
Baseline results path: /workspace/artifacts/gold/pump/baseline/scores/pump__gold__baseline_results.csv


----

In [ ]:
required_context_vars = [
    "CTX",
    "paths",
    "CONFIG",
    "CONFIG_MAP",
    "STAGE_CFG",
    "RESOLVED_PATHS",
    "FILENAMES",
    "VERSIONS_CFG",
    "RUNTIME_CFG",
    "DATASET_CFG",
    "WANDB_CFG",
    "EXECUTION_CFG",
    "PIPELINE",
    "logger",
    "ledger",
    "LOG_PATH",
]

missing_context_vars = [name for name in required_context_vars if name not in globals()]
if missing_context_vars:
    raise NameError(f"Missing required shared context variables: {missing_context_vars}")

logger.info(
    "Context sanity check passed",
    extra={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
)

ledger.add(
    kind="check",
    step="context_sanity_check",
    message="Verified shared notebook context variables are available.",
    data={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
    logger=logger,
)

pd.DataFrame(
    [
        {"name": name, "status": "present"}
        for name in required_context_vars
    ]
)

In [ ]:
gold_required_context_vars = [
    "GOLD_CFG",
]

missing_gold_context_vars = [
    name for name in gold_required_context_vars
    if name not in globals()
]

if missing_gold_context_vars:
    raise NameError(f"Missing Gold context variables: {missing_gold_context_vars}")

logger.info("Gold context sanity check passed")

----

In [5]:
# =========================================================
# Gold baseline artifact directories
# =========================================================

GOLD_BASELINE_ARTIFACT_DIRS = build_artifact_dirs_from_config(
    config=CONFIG,
    stage_key="gold_baseline",
)

GOLD_ARTIFACTS_PATH = GOLD_BASELINE_ARTIFACT_DIRS["stage_dataset_root"]
GOLD_BASELINE_ROOT = GOLD_BASELINE_ARTIFACT_DIRS["root"]
GOLD_BASELINE_CONFIG_DIR = GOLD_BASELINE_ARTIFACT_DIRS["config"]

CONFIG_SNAPSHOT_PATH = (
    GOLD_BASELINE_CONFIG_DIR
    / f"{DATASET_NAME}__gold_baseline__resolved_config.yaml"
)

if CONFIG["execution"].get("save_config_snapshot", True):
    export_config_snapshot(CONFIG, CONFIG_SNAPSHOT_PATH)

BASELINE_MODEL_ARTIFACT_PATH = (
    GOLD_BASELINE_ARTIFACT_DIRS["models"]
    / FILENAMES["baseline_model_file_name"]
)

#BASELINE_MODELS_PATH = BASELINE_MODEL_ARTIFACT_PATH

BASELINE_RESULTS_PATH_CSV = (
    GOLD_BASELINE_ARTIFACT_DIRS["scores"]
    / FILENAMES["baseline_results_file_name_csv"]
)

BASELINE_RESULTS_PATH_PICKLE = (
    GOLD_BASELINE_ARTIFACT_DIRS["scores"]
    / FILENAMES["baseline_results_file_name_pickle"]
)

BASELINE_THRESHOLDS_PATH = (
    GOLD_BASELINE_ARTIFACT_DIRS["thresholds"]
    / FILENAMES["baseline_thresholds_file_name"]
)

BASELINE_SUMMARY_PATH = (
    GOLD_BASELINE_ARTIFACT_DIRS["summaries"]
    / FILENAMES["baseline_summary_file_name"]
)

BASELINE_METADATA_PATH = (
    GOLD_BASELINE_ARTIFACT_DIRS["metadata"]
    / FILENAMES["baseline_metadata_file_name"]
)

baseline_ledger_path = (
    GOLD_BASELINE_ARTIFACT_DIRS["lineage"]
    / FILENAMES["gold_baseline_ledger_file_name"]
)

----

In [6]:
# =============================================================================
# SQL Runtime Context
# Purpose:
#   Establish the Postgres connection and resolve the dataset/run identifiers
#   used by SQL logging and medallion table writes.
# =============================================================================

engine = get_engine_from_env()

CAPSTONE_SCHEMA: str = str(os.getenv("CAPSTONE_SCHEMA", "capstone"))


def first_non_empty_string(*values: object) -> str | None:
    """
    Return the first non-empty string-like value from a list of candidates.

    This helper skips None, empty strings, whitespace-only strings, and
    dictionaries. It is used to resolve dataset/run identifiers without
    accidentally accepting an invalid config object or a placeholder None value.
    """
    for value in values:
        if value is None:
            continue

        if isinstance(value, dict):
            continue

        text_value = str(value).strip()

        if text_value:
            return text_value

    return None


dataset_config = (
    cast(Dict[str, Any], CONFIG.get("dataset", {}))
    if isinstance(CONFIG.get("dataset", {}), dict)
    else {}
)

dataset_config_name = dataset_config.get("name")
dataset_config_id = dataset_config.get("dataset_id", dataset_config.get("id"))
dataset_config_run_id = dataset_config.get("run_id")
dataset_config_asset_id = dataset_config.get("asset_id")

is_synthetic_run = str(RUN_MODE).lower() in {
    "synthetic",
    "synthetic_train",
    "synthetic_run",
    "simulation",
}

if is_synthetic_run:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        os.getenv("SYNTHETIC_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump_synthetic_v1",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        os.getenv("SYNTHETIC_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "synthetic_run_001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        os.getenv("SYNTHETIC_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "pump_asset_001",
    )

else:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "run__001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "asset__001",
    )


if raw_dataset_id is None:
    raise ValueError(
        "DATASET_ID could not be resolved. "
        "Set CAPSTONE_DATASET_ID or configure CONFIG['dataset']['name'] / "
        "CONFIG['dataset']['dataset_id']."
    )

if raw_run_id is None:
    raise ValueError(
        "RUN_ID could not be resolved. "
        "Set CAPSTONE_RUN_ID, CONFIG['dataset']['run_id'], or default_fallbacks.run_id."
    )

if raw_asset_id is None:
    raise ValueError(
        "ASSET_ID could not be resolved. "
        "Set CAPSTONE_ASSET_ID, CONFIG['dataset']['asset_id'], or default_fallbacks.asset_id."
    )


DATASET_ID: str = raw_dataset_id
RUN_ID: str = raw_run_id
ASSET_ID: str = raw_asset_id


print(f"SQL schema: {CAPSTONE_SCHEMA}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Run ID: {RUN_ID}")
print(f"Asset ID: {ASSET_ID}")
print(f"Synthetic run: {is_synthetic_run}")

SQL schema: capstone
Dataset ID: pump
Run ID: run__001
Asset ID: asset__001
Synthetic run: False


#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [7]:

# =============================================================================
# SQL Smoke Check
# Purpose:
#   Confirm the database connection and expected capstone schemas/tables exist.
# =============================================================================

sql_smoke_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN (:capstone_schema, 'bronze', 'silver', 'gold', 'metadata')
    ORDER BY table_schema, table_name
    """,
    params={"capstone_schema": CAPSTONE_SCHEMA},
)

display(sql_smoke_check_dataframe)

,table_schema,table_name
0,bronze,sensor_observations
1,capstone,bronze_observations_input_stage
2,capstone,data_quality_events
3,capstone,pipeline_artifacts
4,capstone,pipeline_runs
5,capstone,simulation_state_control
6,capstone,simulation_timing_config
7,capstone,synthetic_observations_premelt_stage
8,capstone,synthetic_observations_timestamped_stage
9,capstone,synthetic_pump_stream


----

## Start Logging for the Baseline Modeling Stage

Before I load the modeling inputs, I want logging turned on so this notebook records what happened during the run.

This helps with debugging, lineage tracking, and basic pipeline discipline. If I need to confirm what inputs were used or where a step failed, the log gives me a cleaner record than notebook output alone.

In [8]:
log_layer_paths(paths, current_layer=CONTEXT_LAYER, logger=logger)

logger.info(
    "Project paths logged for %s layer",
    CONTEXT_LAYER,
    extra={"stage": CONTEXT_STAGE, "layer": CONTEXT_LAYER},
)

ledger.add(
    kind="step",
    step=f"{CONTEXT_LAYER}_paths_logged",
    message="Logged project layer paths.",
    data={
        "stage": CONTEXT_STAGE,
        "layer": CONTEXT_LAYER,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-15 00:57:16,318 | INFO | capstone.gold.baseline | Project Root Path Loaded: /workspace
2026-06-15 00:57:16,322 | INFO | capstone.gold.baseline | Project Logging Path Loaded: /workspace/logs
2026-06-15 00:57:16,328 | INFO | capstone.gold.baseline | Project Artifacts Path Loaded: /workspace/artifacts
2026-06-15 00:57:16,331 | INFO | capstone.gold.baseline | Project Notebooks Path Loaded: /workspace/notebooks
2026-06-15 00:57:16,333 | INFO | capstone.gold.baseline | Project Truths Path Loaded: /workspace/artifacts/truths
2026-06-15 00:57:16,336 | INFO | capstone.gold.baseline | Project Data Path Loaded: /workspace/data
2026-06-15 00:57:16,341 | INFO | capstone.gold.baseline | Previous Layer (Silver) Path Loaded: /workspace/data/silver
2026-06-15 00:57:16,343 | INFO | capstone.gold.baseline | Previous Layer (Silver) Training Path Loaded: /workspace/data/silver/train
2026-06-15 00:57:16,344 | INFO | capstone.gold.baseline | Previous Layer (Silver) Testing Path Loaded: /workspace/dat

{'ts_utc': '2026-06-15T00:57:16.352409+00:00',
 'stage': 'gold_baseline',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'gold_paths_logged',
 'message': 'Logged project layer paths.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'gold_baseline',
  'layer': 'gold',
  'log_path': '/workspace/logs/gold_modeling_baseline.log'}}

In [9]:
""" 
# Original Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_baseline.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="gold", logger=logger)
 """

' \n# Original Logging Setup\n\n# Create gold log path \ngold_log_path = paths.logs / "gold_modeling_baseline.log"\n\n# Initial Logger\nconfigure_logging(\n    "capstone",\n    gold_log_path,\n    level=logging.DEBUG,\n    overwrite_handlers=True,\n)\n\n# Initiate Logger and log file\nlogger = logging.getLogger("capstone.gold")\n\n# Log load and initiation\nlogger.info("Gold Modeling stage starting")\n\n# Log paths loads\nlog_layer_paths(paths, current_layer="gold", logger=logger)\n '

----

## Initialize Experiment Tracking

This step starts the Weights & Biases run for the baseline modeling stage.

I am using this mainly for run tracking and artifact registration. It helps document the baseline configuration, input references, and saved outputs, but it does not change the underlying modeling logic itself.

In [10]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_baseline",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "train_fraction": TRAIN_FRACTION,
        "baseline_threshold_percentile": BASELINE_THRESHOLD_PERCENTILE,
        "gold_input_path": str(GOLD_FIT_DATA_PATH),
    },
)
logger.info("W&B initialized: %s", wandb_run.name)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: dcoo230 (dcoo230-western-governors-university). Use `wandb login --relogin` to force relogin


2026-06-15 00:57:21,040 | INFO | capstone.gold.baseline | W&B initialized: gold__001


----

## Initialize the Baseline Ledger

Here I create the ledger that tracks the main steps taken during the baseline notebook.

I treat the ledger as a structured run record. It gives me a cleaner summary of the modeling workflow than relying only on printed notebook output, especially when I need to review or compare runs later.

In [11]:
# Ledger was initialized by load_notebook_context().
# Keep this cell as a visible notebook checkpoint instead of re-creating Ledger.

ledger.add(
    kind="step",
    step="ledger_context_ready",
    message="Ledger is available from shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)

logger.info(
    "Ledger ready from notebook context",
    extra={"stage": CONTEXT_STAGE, "recipe_id": CONTEXT_RECIPE_ID},
)


2026-06-15 00:57:21,440 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T00:57:21.440492+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'ledger_context_ready', 'message': 'Ledger is available from shared notebook context.', 'why': None, 'consequence': None, 'data': {'stage': 'gold_baseline', 'recipe_id': 'gold_modeling__v001_baseline', 'log_path': '/workspace/logs/gold_modeling_baseline.log'}}
2026-06-15 00:57:21,444 | INFO | capstone.gold.baseline | Ledger ready from notebook context


In [12]:
""" 
# Original Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)
 """

' \n# Original Ledger Setup\n\nledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)\n\nledger.add(\n    kind="step",\n    step="init",\n    message="Initialized ledger",\n    logger=logger\n)\n '

----

## Load Gold Modeling Inputs and Resolve the Parent Truth

This is the point where I load the Gold preprocessing outputs that the baseline model depends on.

In this step I am:
- loading the scaled Gold dataframe
- resolving the parent Gold truth record
- confirming the dataset identity
- substituting truth-linked artifact paths when needed
- loading the Gold fit dataframe
- loading the Stage 1 feature list

This matters because the baseline notebook should not invent its own inputs. It should inherit the prepared modeling artifacts that were created in Gold preprocessing.

In [13]:
logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_SCALED_DATA_PATH)
gold_preprocessed_scaled_dataframe = load_data(GOLD_PREPROCESSED_SCALED_DATA_PATH)

GOLD_DATASET_NAME = (
    gold_preprocessed_scaled_dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)
GOLD_DATASET_NAME = GOLD_DATASET_NAME[GOLD_DATASET_NAME != ""]

if len(GOLD_DATASET_NAME) == 0:
    raise ValueError("Gold baseline input dataframe is missing usable meta__dataset values.")

GOLD_DATASET_NAME = str(GOLD_DATASET_NAME.iloc[0]).strip()

gold_truth = load_parent_truth_record_from_dataframe(
    dataframe=gold_preprocessed_scaled_dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="gold",
    dataset_name=GOLD_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(gold_truth)
GOLD_PARENT_TRUTH_HASH = get_truth_hash(gold_truth)

PARENT_PIPELINE_MODE = get_pipeline_mode_from_truth(gold_truth)
if PARENT_PIPELINE_MODE is not None:
    PIPELINE_MODE = PARENT_PIPELINE_MODE

GOLD_TRUTH_PATH = (
    TRUTHS_PATH
    / "gold"
    / f"{DATASET_NAME}__gold__truth__{GOLD_PARENT_TRUTH_HASH}.json"
)

if not isinstance(gold_truth, dict):
    raise TypeError(
        "Gold parent truth record must be a dictionary. "
        f"Got {type(gold_truth).__name__}: {gold_truth!r}"
    )

raw_gold_truth_runtime_facts = gold_truth.get("runtime_facts", {})
raw_gold_truth_artifact_paths = gold_truth.get("artifact_paths", {})

gold_truth_runtime_facts: dict[str, Any] = (
    raw_gold_truth_runtime_facts
    if isinstance(raw_gold_truth_runtime_facts, dict)
    else {}
)

gold_truth_artifact_paths: dict[str, Any] = (
    raw_gold_truth_artifact_paths
    if isinstance(raw_gold_truth_artifact_paths, dict)
    else {}
)

GOLD_FIT_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_fit_path", str(GOLD_FIT_DATA_PATH)))
STAGE1_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage1_features_path", str(STAGE1_FEATURES_PATH)))

logger.info("Resolved Gold baseline dataset name from Gold truth: %s", DATASET_NAME)
logger.info("Resolved Gold truth path: %s", GOLD_TRUTH_PATH)
logger.info("Resolved Gold fit parquet from Gold truth: %s", GOLD_FIT_DATA_PATH)
logger.info("Resolved Stage 1 features path from Gold truth: %s", STAGE1_FEATURES_PATH)

logger.info("Loading Gold fit parquet: %s", GOLD_FIT_DATA_PATH)
gold_fit_dataframe = load_data(GOLD_FIT_DATA_PATH)

logger.info("Loading Stage 1 features JSON: %s", STAGE1_FEATURES_PATH)
raw_stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)

if raw_stage1_feature_columns is None:
    raise ValueError(f"Stage 1 features file returned None: {STAGE1_FEATURES_PATH}")

if not isinstance(raw_stage1_feature_columns, (list, tuple)):
    raise TypeError(
        "Stage 1 features JSON must contain a list of column names. "
        f"Got {type(raw_stage1_feature_columns).__name__}: {raw_stage1_feature_columns!r}"
    )

stage1_feature_columns: list[str] = [
    str(column_name).strip()
    for column_name in raw_stage1_feature_columns
    if str(column_name).strip()
]

if not stage1_feature_columns:
    raise ValueError(f"Stage 1 features list is empty: {STAGE1_FEATURES_PATH}")

print("Gold baseline dataset name from parent truth:", DATASET_NAME)
print("Gold baseline parent truth hash:", GOLD_PARENT_TRUTH_HASH)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded Gold scaled parquet, loaded Gold truth, substituted truth-linked artifact paths, then loaded baseline inputs.",
    data={
        "gold_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
        "gold_fit_shape": list(gold_fit_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
    },
    logger=logger,
)

display(gold_fit_dataframe.head(3))

2026-06-15 00:57:22,142 | INFO | capstone.gold.baseline | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-06-15 00:57:22,152 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-06-15 00:57:24,090 | INFO | capstone.gold.baseline | Resolved Gold baseline dataset name from Gold truth: pump
2026-06-15 00:57:24,093 | INFO | capstone.gold.baseline | Resolved Gold truth path: /workspace/artifacts/truths/gold/pump__gold__truth__1b39755619d21614721b74a063b90aceddebaa1f4b857393db937f146eccbec5.json
2026-06-15 00:57:24,095 | INFO | capstone.gold.baseline | Resolved Gold fit parquet from Gold truth: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-06-15 00:57:24,096 | INFO | capstone.gold.baseline | Resolved Stage 1 features path from Gold truth: /workspace/artifacts/gold/pump/preprocessing/features/pump__gold__stage1_features.json
2026-06-15 00:57:24,098 | INFO | capstone.g

Gold baseline dataset name from parent truth: pump
Gold baseline parent truth hash: 1b39755619d21614721b74a063b90aceddebaa1f4b857393db937f146eccbec5


,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_43__value_abnormal_flag,sensor_43__delta_abnormal_flag,sensor_43__any_abnormal_flag,sensor_44__value_deviation,sensor_44__delta_deviation,sensor_44__value_abnormal_flag,sensor_44__delta_abnormal_flag,sensor_44__any_abnormal_flag,sensor_45__value_deviation,sensor_45__delta_deviation,sensor_45__value_abnormal_flag,sensor_45__delta_abnormal_flag,sensor_45__any_abnormal_flag,sensor_46__value_deviation,sensor_46__delta_deviation,sensor_46__value_abnormal_flag,sensor_46__delta_abnormal_flag,sensor_46__any_abnormal_flag,sensor_47__value_deviation,sensor_47__delta_deviation,sensor_47__value_abnormal_flag,sensor_47__delta_abnormal_flag,sensor_47__any_abnormal_flag,sensor_48__value_deviation,sensor_48__delta_deviation,sensor_48__value_abnormal_flag,sensor_48__delta_abnormal_flag,sensor_48__any_abnormal_flag,sensor_49__value_deviation,sensor_49__delta_deviation,sensor_49__value_abnormal_flag,sensor_49__delta_abnormal_flag,sensor_49__any_abnormal_flag,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled,meta__row_id,meta__is_train_flag
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,14598431322315673869,run__001,sensor.csv,0,fit_normal_only,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,False,False,0.076923,NaN,False,False,False,4.932429,NaN,False,False,False,0.980392,NaN,False,False,False,0.739131,NaN,False,False,False,0.716570,NaN,False,False,False,2.780487,NaN,False,False,False,0.275862,NaN,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,0,True
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,15954729095895098000,run__001,sensor.csv,1,fit_normal_only,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,False,False,0.076923,0.000000,False,False,False,4.932429,0.000000,False,False,False,0.980392,0.000000,False,False,False,0.739131,0.000000,False,False,False,0.716570,0.000000,False,False,False,2.780487,0.000000,False,False,False,0.275862,0.000000,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,1,True
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,bat

----

## Validate Stable Row Identity for Baseline Scoring

Before baseline scoring begins, I want to confirm that the Gold preprocessing stage already stamped a stable row identifier that can be used to attach row-level anomaly outputs back to the full dataset.

In [14]:
gold_preprocessed_scaled_dataframe = ensure_stable_row_id(
    gold_preprocessed_scaled_dataframe,
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="validate_baseline_row_tracking",
    message="Validated stable row identity on Gold baseline modeling input dataframe.",
    data={
        "row_id_column": "meta__row_id",
        "row_count": int(len(gold_preprocessed_scaled_dataframe)),
        "row_id_unique": bool(gold_preprocessed_scaled_dataframe["meta__row_id"].is_unique),
    },
    logger=logger,
)

2026-06-15 00:57:25,629 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T00:57:25.628984+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'validate_baseline_row_tracking', 'message': 'Validated stable row identity on Gold baseline modeling input dataframe.', 'why': None, 'consequence': None, 'data': {'row_id_column': 'meta__row_id', 'row_count': 220320, 'row_id_unique': True}}


{'ts_utc': '2026-06-15T00:57:25.628984+00:00',
 'stage': 'gold_baseline',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'validate_baseline_row_tracking',
 'message': 'Validated stable row identity on Gold baseline modeling input dataframe.',
 'why': None,
 'consequence': None,
 'data': {'row_id_column': 'meta__row_id',
  'row_count': 220320,
  'row_id_unique': True}}

----

## Build the Train and Test Masks

Before fitting or evaluating the model, I need to recover the train/test split that was already stamped during Gold preprocessing.

The key idea here is that the baseline notebook is **not** creating a new split. It is reusing the split created upstream so the modeling and comparison stages all work from the same partition.

In [15]:
# Masks must exist in scaled parquet.
if "meta__is_train_flag" not in gold_preprocessed_scaled_dataframe.columns:
    raise ValueError(
        "meta__is_train_flag missing from gold_preprocessed_scaled_dataframe. "
        "Gold preprocessing must stamp it before saving."
    )

train_mask: pd.Series = (
    gold_preprocessed_scaled_dataframe["meta__is_train_flag"]
    .fillna(False)
    .astype(bool)
)

test_mask: pd.Series = (~train_mask).astype(bool)

train_mask_array: np.ndarray = train_mask.to_numpy(dtype=bool)
test_mask_array: np.ndarray = test_mask.to_numpy(dtype=bool)

logger.info(
    "Split counts: all=%d train=%d test=%d",
    len(train_mask),
    int(train_mask.sum()),
    int(test_mask.sum()),
)

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    logger.info(
        "Anomalies: all=%d test=%d",
        int(
            gold_preprocessed_scaled_dataframe["anomaly_flag"]
            .fillna(0)
            .astype(int)
            .sum()
        ),
        int(
            gold_preprocessed_scaled_dataframe.loc[test_mask, "anomaly_flag"]
            .fillna(0)
            .astype(int)
            .sum()
        ),
    )

2026-06-15 00:57:25,930 | INFO | capstone.gold.baseline | Split counts: all=220320 train=136431 test=83889
2026-06-15 00:57:25,938 | INFO | capstone.gold.baseline | Anomalies: all=14484 test=118


### Ask

Why am I reusing the saved split instead of making a fresh split inside this notebook?

### Answer

I want the baseline results to stay aligned with the rest of the Gold workflow.

If this notebook made its own split, then the baseline, cascade, and comparison notebooks could end up evaluating different row partitions. That would make the comparison less reliable.

So this step is really about consistency. The split belongs to Gold preprocessing, and the model notebooks should reuse it.

----

## Prepare Labels and Feature Matrices

Now I prepare the arrays that the baseline model will use.

This includes:
- the available anomaly labels for evaluation
- the test-only labels
- the fit feature matrix built from the Gold fit dataframe
- the full feature matrix built from the scaled Gold dataframe

A detail that matters here is that the model is fit on the **fit-normal-only** subset, while scoring is done across the **full scaled Gold dataframe**. That means training and scoring are intentionally not using the exact same row set.

In [16]:
all_labels: np.ndarray | None = None
test_labels: np.ndarray | None = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    all_labels = (
        gold_preprocessed_scaled_dataframe["anomaly_flag"]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

    test_labels = (
        gold_preprocessed_scaled_dataframe.loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

# =========================================================
# Build model feature matrices
# =========================================================
# Keep these as DataFrames, not NumPy arrays.
# This prevents the sklearn warning:
# "X has feature names, but IsolationForest was fitted without feature names"

missing_stage1_features = [
    column_name
    for column_name in stage1_feature_columns
    if column_name not in gold_preprocessed_scaled_dataframe.columns
]

if missing_stage1_features:
    raise ValueError(
        "Stage 1 feature columns are missing from gold_preprocessed_scaled_dataframe:\n"
        f"{missing_stage1_features[:25]}"
    )

missing_fit_features = [
    column_name
    for column_name in stage1_feature_columns
    if column_name not in gold_fit_dataframe.columns
]

if missing_fit_features:
    raise ValueError(
        "Stage 1 feature columns are missing from gold_fit_dataframe:\n"
        f"{missing_fit_features[:25]}"
    )

baseline_train_fit_features: pd.DataFrame = gold_fit_dataframe.loc[
    :,
    stage1_feature_columns,
].copy()

baseline_all_features: pd.DataFrame = gold_preprocessed_scaled_dataframe.loc[
    :,
    stage1_feature_columns,
].copy()

baseline_test_features: pd.DataFrame = gold_preprocessed_scaled_dataframe.loc[
    test_mask,
    stage1_feature_columns,
].copy()

### Ask

Why am I fitting on the fit subset but scoring the full Gold dataset?

### Answer

Because those two steps are serving different purposes.

The fit subset is meant to represent the cleaner normal training behavior that the unsupervised model should learn from. The full Gold dataframe is what I want to evaluate and inspect after the model has been trained.

So the logic is:
- **fit on the normal reference behavior**
- **score the full dataset to see where alerts appear**
- **evaluate the scoring outcome on the held-out test rows**

----

## Define Baseline Scoring, Thresholding, and Evaluation Helpers

These helper functions break the baseline workflow into smaller pieces:
- computing anomaly scores from the Isolation Forest
- choosing a score threshold by percentile
- evaluating the thresholded predictions against available labels

I like separating these steps because it makes the notebook easier to read and easier to compare later when I move into the cascade notebooks.

In [17]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: pd.DataFrame | np.ndarray,
) -> np.ndarray:
    """
    Compute anomaly scores from a fitted IsolationForest model.

    Higher returned values mean more anomalous.

    Notes
    -----
    The feature_matrix can be a DataFrame or NumPy array, but for this
    project we prefer DataFrames so sklearn keeps feature-name tracking.
    """
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores

    return anomaly_scores

#### Define percentile threshold selection

This cell defines helper logic used by the surrounding `Define Baseline Scoring, Thresholding, and Evaluation Helpers` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [18]:
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))

#### Define label-based model evaluation

This cell defines helper logic used by the surrounding `Define Baseline Scoring, Thresholding, and Evaluation Helpers` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [19]:
def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict[str, float | None]:
    true_labels_array: np.ndarray = np.asarray(true_labels, dtype=int)
    anomaly_scores_array: np.ndarray = np.asarray(anomaly_scores, dtype=float)

    predicted_labels = (anomaly_scores_array >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels_array,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results: dict[str, float | None] = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels_array)) == 2:
        results["roc_auc"] = float(
            roc_auc_score(true_labels_array, anomaly_scores_array)
        )
        results["pr_auc"] = float(
            average_precision_score(true_labels_array, anomaly_scores_array)
        )
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

## Fit the Baseline Isolation Forest Model

This is the actual model-fitting step for the baseline notebook.

Here I initialize the Isolation Forest using the configured estimator count and random seed, then fit it on the Gold fit feature matrix. Since that fit matrix comes from the normal-only Gold fit dataframe, the model is being trained on the normal reference subset rather than on the full mixed dataset.

In [20]:
baseline_model = IsolationForest(
    n_estimators=BASELINE_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)


baseline_model.fit(baseline_train_fit_features)


IsolationForest(n_estimators=200, n_jobs=-1, random_state=42)

## Score the Dataset, Set the Threshold, and Build Baseline Results

After fitting the model, I score both:
- the normal-only fit subset
- the full scaled Gold dataframe

Then I use the training-score distribution to choose the anomaly threshold by percentile. After that, I apply the threshold to the full dataset scores and create the baseline results dataframe with:
- `baseline_score`
- `baseline_flag`

This is one of the most important steps in the notebook because it converts the fitted unsupervised model into row-level alert output that I can evaluate and save.

### Ask

Why am I choosing the threshold from the training-score distribution instead of from the test data?

### Answer

Because I do not want the test set deciding the baseline rule.

The threshold should come from the model's view of the normal training behavior, not from the held-out rows that I later use for evaluation. If I tuned the threshold on the test set, that would blur the line between training and evaluation.

So the logic here is:
- fit the model on the normal reference subset
- use that score distribution to choose the threshold
- apply the threshold to all rows
- evaluate the outcome on the test rows

In [21]:
# Score normal-only training rows.
baseline_train_scores: np.ndarray = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_train_fit_features,
)

# Score all rows.
baseline_all_scores: np.ndarray = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_all_features,
)

if len(baseline_all_scores) != len(gold_preprocessed_scaled_dataframe):
    raise ValueError(
        "Score length mismatch vs all-rows dataframe. Check feature matrix source."
    )

baseline_threshold = choose_threshold_by_percentile(
    baseline_train_scores,
    BASELINE_THRESHOLD_PERCENTILE,
)

baseline_flags: np.ndarray = (
    baseline_all_scores >= baseline_threshold
).astype(int)

baseline_test_scores: np.ndarray = baseline_all_scores[test_mask_array]
baseline_test_flags: np.ndarray = baseline_flags[test_mask_array]

baseline_results = gold_preprocessed_scaled_dataframe.copy()
baseline_results["baseline_score"] = baseline_all_scores
baseline_results["baseline_flag"] = baseline_flags

baseline_metrics: dict[str, Any] = {
    "model": "Baseline IsolationForest",
    "threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
    "threshold": float(baseline_threshold),
    "alert_count_all_rows": int(baseline_flags.sum()),
    "alert_count_test_rows": int(baseline_test_flags.sum()),
}

if test_labels is not None:
    test_labels_array: np.ndarray = np.asarray(test_labels, dtype=int)

    baseline_metrics.update(
        evaluate_against_labels(
            test_labels_array,
            baseline_test_scores,
            baseline_threshold,
        )
    )

ledger.add(
    kind="step",
    step="run_baseline_isolation_forest",
    message="Ran baseline Isolation Forest fit on normal-only rows and scored the full scaled Gold dataset; evaluated on test rows.",
    data={
        "estimator_count": int(BASELINE_ESTIMATOR_COUNT),
        "threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
        "threshold": float(baseline_threshold),
        "training_rows": int(len(gold_fit_dataframe)),
        "test_rows": int(test_mask.sum()),
        "all_rows": int(len(gold_preprocessed_scaled_dataframe)),
        "train_rows": int(train_mask.sum()),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_test_rows": int(baseline_test_flags.sum()),
        "precision": baseline_metrics.get("precision"),
        "recall": baseline_metrics.get("recall"),
        "f1": baseline_metrics.get("f1"),
        "roc_auc": baseline_metrics.get("roc_auc"),
        "pr_auc": baseline_metrics.get("pr_auc"),
    },
    logger=logger,
)

display(baseline_metrics)

2026-06-15 00:57:30,858 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T00:57:30.858869+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'run_baseline_isolation_forest', 'message': 'Ran baseline Isolation Forest fit on normal-only rows and scored the full scaled Gold dataset; evaluated on test rows.', 'why': None, 'consequence': None, 'data': {'estimator_count': 200, 'threshold_percentile': 97.0, 'threshold': 0.5040984526829705, 'training_rows': 77395, 'test_rows': 83889, 'all_rows': 220320, 'train_rows': 136431, 'feature_count': 50, 'alert_count_test_rows': 31200, 'precision': 0.003782051282051282, 'recall': 1.0, 'f1': 0.00753560252889712, 'roc_auc': 0.9406852498811833, 'pr_auc': 0.12201014363272505}}


{'model': 'Baseline IsolationForest',
 'threshold_percentile': 97.0,
 'threshold': 0.5040984526829705,
 'alert_count_all_rows': 87735,
 'alert_count_test_rows': 31200,
 'precision': 0.003782051282051282,
 'recall': 1.0,
 'f1': 0.00753560252889712,
 'roc_auc': 0.9406852498811833,
 'pr_auc': 0.12201014363272505}

### Ask

What should I pay attention to in the baseline metrics here?

### Answer

The main things I care about are:
- the chosen threshold percentile and actual threshold value
- how many alerts were raised across all rows
- how many alerts appeared in the test rows
- precision, recall, and F1 when test labels are available
- ROC AUC and PR AUC when both classes are present in the evaluation labels

This is the first benchmark for the project. So I am less worried about perfection here and more focused on creating a clean, traceable baseline that the cascade model can later be compared against.

----

## Validate Baseline Feature Columns

### Ask

Why check the baseline feature columns before saving results?

### Answer

The baseline model should use the same prepared Stage 1 feature list created during Gold preprocessing.

This cell confirms that the expected baseline feature columns are present in the scaled Gold dataframe. That prevents the notebook from producing results with missing or misaligned model inputs.

In [22]:
# =========================================================
# Score baseline with row tracking
# =========================================================
# Important:
# score_isolation_forest_stage() returns model.score_samples() as the helper
# score column. For IsolationForest, score_samples is higher for more normal
# rows and lower for more anomalous rows.
#
# The project baseline score uses the opposite direction:
#     baseline_score = -model.score_samples(...)
#
# So this cell preserves the helper's raw score_samples output separately
# and restores baseline_score to the project-defined anomaly score.

source_gold_dataframe: pd.DataFrame = gold_preprocessed_scaled_dataframe

baseline_feature_columns: list[str] = list(stage1_feature_columns)

missing_baseline_feature_columns = [
    column_name
    for column_name in baseline_feature_columns
    if column_name not in source_gold_dataframe.columns
]

if missing_baseline_feature_columns:
    raise ValueError(
        "Some baseline feature columns are missing from gold_preprocessed_scaled_dataframe:\n"
        f"{missing_baseline_feature_columns[:25]}"
    )

baseline_stage_input_df = build_stage_scoring_frame(
    dataframe=source_gold_dataframe,
    feature_columns=baseline_feature_columns,
    mask=None,
    row_id_column="meta__row_id",
)

baseline_stage_results_df = score_isolation_forest_stage(
    stage_dataframe=baseline_stage_input_df,
    model=baseline_model,
    feature_columns=baseline_feature_columns,
    stage_name="baseline",
    row_id_column="meta__row_id",
)

# Rename the helper outputs so the raw IsolationForest score does not replace
# the project-level anomaly score.
baseline_stage_results_df = baseline_stage_results_df.rename(
    columns={
        "baseline_score": "baseline_score_samples_raw",
        "baseline_flag": "baseline_predict_flag",
    }
)

baseline_row_tracking_columns = [
    "meta__row_id",
    "baseline_score_samples_raw",
    "baseline_decision",
    "baseline_pred",
    "baseline_predict_flag",
]

baseline_row_tracking_columns = [
    column_name
    for column_name in baseline_row_tracking_columns
    if column_name in baseline_stage_results_df.columns
]

# Drop old baseline columns first so rerunning this cell does not create
# duplicate _x / _y columns.
scored_gold_dataframe: pd.DataFrame = source_gold_dataframe.drop(
    columns=[
        "baseline_score_samples_raw",
        "baseline_decision",
        "baseline_pred",
        "baseline_predict_flag",
        "baseline_score",
        "baseline_flag",
    ],
    errors="ignore",
)

scored_gold_dataframe = scored_gold_dataframe.merge(
    baseline_stage_results_df[baseline_row_tracking_columns],
    on="meta__row_id",
    how="left",
)

# Recompute the canonical project anomaly score directly from the model.
# Higher values mean more anomalous.
baseline_all_scores = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_all_features,
)

baseline_all_decisions = baseline_model.decision_function(baseline_all_features)
baseline_all_preds = baseline_model.predict(baseline_all_features)

if len(baseline_all_scores) != len(scored_gold_dataframe):
    raise ValueError(
        "baseline_all_scores length does not match gold_preprocessed_scaled_dataframe."
    )

scored_gold_dataframe["baseline_score"] = baseline_all_scores
scored_gold_dataframe["baseline_decision"] = baseline_all_decisions
scored_gold_dataframe["baseline_pred"] = baseline_all_preds
scored_gold_dataframe["baseline_flag"] = (
    scored_gold_dataframe["baseline_score"] >= baseline_threshold
).astype(int)

# Avoid Pylance's notebook-cell unbound-variable warning while still updating
# the shared notebook variable for later cells.
globals()["gold_preprocessed_scaled_dataframe"] = scored_gold_dataframe

ledger.add(
    kind="step",
    step="score_baseline_with_row_tracking",
    message="Scored the full Gold dataframe with baseline row-level tracking and preserved project-defined anomaly score direction.",
    data={
        "row_count_scored": int(len(baseline_stage_results_df)),
        "baseline_threshold": float(baseline_threshold),
        "baseline_flag_count": int(scored_gold_dataframe["baseline_flag"].sum()),
        "baseline_flag_count_test_rows": int(
            scored_gold_dataframe.loc[test_mask, "baseline_flag"].sum()
        ),
        "score_column": "baseline_score",
        "raw_score_samples_column": "baseline_score_samples_raw",
        "decision_column": "baseline_decision",
        "pred_column": "baseline_pred",
        "flag_column": "baseline_flag",
    },
    logger=logger,
)

print("Baseline row-tracked scoring complete.")
print(
    {
        "baseline_threshold": float(baseline_threshold),
        "baseline_alert_count_all_rows": int(scored_gold_dataframe["baseline_flag"].sum()),
        "baseline_alert_count_test_rows": int(
            scored_gold_dataframe.loc[test_mask, "baseline_flag"].sum()
        ),
        "baseline_score_min": float(scored_gold_dataframe["baseline_score"].min()),
        "baseline_score_median": float(scored_gold_dataframe["baseline_score"].median()),
        "baseline_score_max": float(scored_gold_dataframe["baseline_score"].max()),
    }
)

2026-06-15 00:57:40,954 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T00:57:40.954020+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'score_baseline_with_row_tracking', 'message': 'Scored the full Gold dataframe with baseline row-level tracking and preserved project-defined anomaly score direction.', 'why': None, 'consequence': None, 'data': {'row_count_scored': 220320, 'baseline_threshold': 0.5040984526829705, 'baseline_flag_count': 87735, 'baseline_flag_count_test_rows': 31200, 'score_column': 'baseline_score', 'raw_score_samples_column': 'baseline_score_samples_raw', 'decision_column': 'baseline_decision', 'pred_column': 'baseline_pred', 'flag_column': 'baseline_flag'}}


Baseline row-tracked scoring complete.
{'baseline_threshold': 0.5040984526829705, 'baseline_alert_count_all_rows': 87735, 'baseline_alert_count_test_rows': 31200, 'baseline_score_min': 0.35608122465682657, 'baseline_score_median': 0.4821946872241749, 'baseline_score_max': 0.7451612130179065}


#### Define label-based model evaluation

This cell evaluates model behavior against the configured labels or thresholds. The resulting counts and metrics should match the saved JSON/CSV summaries used in Task 3.

In [23]:
# =========================================================
# Synchronize baseline_results after row-tracked scoring
# =========================================================

baseline_results = gold_preprocessed_scaled_dataframe.copy()

# Keep the project-defined percentile threshold as the actual alert rule.
baseline_results["baseline_score"] = baseline_all_scores
baseline_results["baseline_threshold"] = float(baseline_threshold)
baseline_results["baseline_threshold_percentile"] = float(BASELINE_THRESHOLD_PERCENTILE)
baseline_results["baseline_flag"] = (
    baseline_results["baseline_score"] >= baseline_threshold
).astype(int)

baseline_alert_count_all_rows = int(baseline_results["baseline_flag"].sum())
baseline_alert_count_test_rows = int(
    baseline_results.loc[test_mask, "baseline_flag"].sum()
)

if not isinstance(baseline_metrics, dict):
    baseline_metrics = {}

if test_labels is not None:
    test_labels_array = np.asarray(test_labels, dtype=int)
    baseline_test_scores = baseline_results.loc[
        test_mask,
        "baseline_score",
    ].to_numpy(dtype=float)

    baseline_metrics.update(
        evaluate_against_labels(
            test_labels_array,
            baseline_test_scores,
            baseline_threshold,
        )
    )

if baseline_alert_count_all_rows == 0:
    raise ValueError(
        "Baseline produced zero alerts after synchronization. "
        "This usually means baseline_score is using raw score_samples instead "
        "of the project anomaly score direction (-score_samples)."
    )

baseline_metrics["alert_count_all_rows"] = baseline_alert_count_all_rows
baseline_metrics["alert_count_test_rows"] = baseline_alert_count_test_rows

ledger.add(
    kind="step",
    step="synchronize_baseline_results_after_row_tracking",
    message="Synchronized baseline_results after row-tracked scoring and reapplied the percentile-threshold alert rule.",
    data={
        "baseline_threshold": float(baseline_threshold),
        "baseline_threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
        "alert_count_all_rows": baseline_alert_count_all_rows,
        "alert_count_test_rows": baseline_alert_count_test_rows,
        "precision": baseline_metrics.get("precision"),
        "recall": baseline_metrics.get("recall"),
        "f1": baseline_metrics.get("f1"),
    },
    logger=logger,
)

display(baseline_metrics)

2026-06-15 00:57:42,386 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T00:57:42.386392+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'synchronize_baseline_results_after_row_tracking', 'message': 'Synchronized baseline_results after row-tracked scoring and reapplied the percentile-threshold alert rule.', 'why': None, 'consequence': None, 'data': {'baseline_threshold': 0.5040984526829705, 'baseline_threshold_percentile': 97.0, 'alert_count_all_rows': 87735, 'alert_count_test_rows': 31200, 'precision': 0.003782051282051282, 'recall': 1.0, 'f1': 0.00753560252889712}}


{'model': 'Baseline IsolationForest',
 'threshold_percentile': 97.0,
 'threshold': 0.5040984526829705,
 'alert_count_all_rows': 87735,
 'alert_count_test_rows': 31200,
 'precision': 0.003782051282051282,
 'recall': 1.0,
 'f1': 0.00753560252889712,
 'roc_auc': 0.9406852498811833,
 'pr_auc': 0.12201014363272505}

#### Define baseline output validation checks

This cell defines helper logic used by the surrounding `Validate Baseline Feature Columns` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [24]:
def validate_baseline_output(
    results_dataframe: pd.DataFrame,
    *,
    test_mask: pd.Series,
    label_column: str = "anomaly_flag",
    flag_column: str = "baseline_flag",
    score_column: str = "baseline_score",
    row_id_column: str = "meta__row_id",
    episode_column: str = "meta__episode_id",
) -> dict:
    required_columns = [
        row_id_column,
        flag_column,
        score_column,
        "meta__is_train_flag",
    ]

    for column_name in required_columns:
        if column_name not in results_dataframe.columns:
            raise ValueError(f"Missing required baseline output column: {column_name}")

    if len(test_mask) != len(results_dataframe):
        raise ValueError("test_mask length does not match baseline results dataframe.")

    if not results_dataframe[row_id_column].is_unique:
        raise ValueError(f"{row_id_column} is not unique in baseline results.")

    if results_dataframe[score_column].isna().any():
        raise ValueError(f"{score_column} contains missing values.")

    invalid_flags = sorted(set(results_dataframe[flag_column].dropna().unique()) - {0, 1})

    if invalid_flags:
        raise ValueError(f"{flag_column} contains non-binary values: {invalid_flags}")

    test_dataframe = results_dataframe.loc[test_mask].copy()

    validation_summary = {
        "all_rows": int(len(results_dataframe)),
        "test_rows": int(len(test_dataframe)),
        "alert_count_all_rows": int(results_dataframe[flag_column].sum()),
        "alert_count_test_rows": int(test_dataframe[flag_column].sum()),
        "test_episode_count": (
            int(test_dataframe[episode_column].nunique())
            if episode_column in test_dataframe.columns
            else None
        ),
    }

    if label_column in results_dataframe.columns:
        y_true = test_dataframe[label_column].fillna(0).astype(int).to_numpy()
        y_pred = test_dataframe[flag_column].fillna(0).astype(int).to_numpy()

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        validation_summary.update(
            {
                "test_anomaly_rows": int(y_true.sum()),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            }
        )

    print("Baseline output validation passed.")
    return validation_summary



#### Define baseline output validation checks

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [25]:

baseline_output_validation = validate_baseline_output(
    baseline_results,
    test_mask=test_mask,
)

ledger.add(
    kind="step",
    step="validate_baseline_output",
    message="Validated baseline scored output on held-out test rows.",
    data=baseline_output_validation,
    logger=logger,
)

display(pd.DataFrame([baseline_output_validation]))

2026-06-15 00:57:43,225 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T00:57:43.225287+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'validate_baseline_output', 'message': 'Validated baseline scored output on held-out test rows.', 'why': None, 'consequence': None, 'data': {'all_rows': 220320, 'test_rows': 83889, 'alert_count_all_rows': 87735, 'alert_count_test_rows': 31200, 'test_episode_count': 3, 'test_anomaly_rows': 118, 'precision': 0.003782051282051282, 'recall': 1.0, 'f1': 0.00753560252889712, 'tn': 52689, 'fp': 31082, 'fn': 0, 'tp': 118}}


Baseline output validation passed.


,all_rows,test_rows,alert_count_all_rows,alert_count_test_rows,test_episode_count,test_anomaly_rows,precision,recall,f1,tn,fp,fn,tp
0,220320,83889,87735,31200,3,118,0.003782,1.0,0.007536,52689,31082,0,118


----

## Build the Baseline Truth Record and Save the Baseline Artifacts

Now I turn the baseline results into a formal pipeline artifact.

This step does several important things:
- summarizes the baseline metrics
- builds the baseline truth record
- stamps lineage columns onto the result dataframe
- links the baseline output back to the parent Gold truth
- saves the results, model, thresholds, summary, metadata, and truth record

At this point, the baseline output becomes more than just notebook output. It becomes a tracked deliverable in the pipeline.

In [26]:
baseline_alert_count_all_rows = int(baseline_results["baseline_flag"].sum())
baseline_alert_count_test_rows = int(baseline_results.loc[test_mask, "baseline_flag"].sum())

baseline_thresholds = {
    "baseline_threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
    "baseline_threshold": float(baseline_threshold),
}

baseline_summary = {
    "dataset_name": DATASET_NAME,
    "baseline_metrics": baseline_metrics,
    "alert_count_all_rows": baseline_alert_count_all_rows,
    "alert_count_test_rows": baseline_alert_count_test_rows,
    "result_row_count": int(len(baseline_results)),
}

truth_config_object = globals().get("TRUTH_CONFIG")
run_mode_value = globals().get("RUN_MODE")
config_profile_value = globals().get("CONFIG_PROFILE", "default")
gold_process_run_id_value = globals().get("GOLD_PROCESS_RUN_ID")

if isinstance(truth_config_object, dict):
    truth_config_snapshot = truth_config_object
else:
    truth_config_snapshot = {
        "runtime": {
            "stage": "gold_baseline",
            "dataset": DATASET_NAME,
            "mode": run_mode_value,
            "profile": config_profile_value,
        }
    }

baseline_truth_layer_name = "gold_baseline"

if isinstance(gold_process_run_id_value, str) and gold_process_run_id_value.strip():
    baseline_process_run_id = gold_process_run_id_value
else:
    baseline_process_run_id = make_process_run_id("gold_baseline_process")

baseline_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=baseline_truth_layer_name,
    process_run_id=baseline_process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
)

baseline_truth = update_truth_section(
    baseline_truth,
    "config_snapshot",
    truth_config_snapshot,
)

baseline_truth = update_truth_section(
    baseline_truth,
    "runtime_facts",
    {
        "baseline_threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
        "baseline_threshold": float(baseline_threshold),
        "baseline_estimator_count": int(BASELINE_ESTIMATOR_COUNT),
        "train_fraction": float(TRAIN_FRACTION),
        "random_seed": int(RANDOM_SEED),
        "alert_count_all_rows": baseline_alert_count_all_rows,
        "alert_count_test_rows": baseline_alert_count_test_rows,
        "result_row_count": int(len(baseline_results)),
        "parent_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_process_run_id": gold_truth.get("process_run_id"),
        "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    },
)

baseline_truth = update_truth_section(
    baseline_truth,
    "artifact_paths",
    {
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_preprocessed_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
        "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
        "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
        "baseline_models_path": str(BASELINE_MODELS_PATH),
        "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
        "baseline_metadata_path": str(BASELINE_METADATA_PATH),
    },
)

baseline_meta_columns = sorted(
    set(
        identify_meta_columns(baseline_results)
        + [
            "meta__truth_hash",
            "meta__parent_truth_hash",
            "meta__pipeline_mode",
        ]
    )
)

baseline_feature_columns = identify_feature_columns(baseline_results)

baseline_truth_record = build_truth_record(
    truth_base=baseline_truth,
    row_count=len(baseline_results),
    column_count=baseline_results.shape[1] + 3,
    meta_columns=baseline_meta_columns,
    feature_columns=baseline_feature_columns,
)

BASELINE_TRUTH_HASH = baseline_truth_record["truth_hash"]

baseline_results = stamp_truth_columns(
    baseline_results,
    truth_hash=BASELINE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

baseline_truth_path = save_truth_record(
    baseline_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=baseline_truth_layer_name,
)

append_truth_index(
    baseline_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

baseline_summary["baseline_truth_hash"] = BASELINE_TRUTH_HASH
baseline_summary["baseline_truth_path"] = str(baseline_truth_path)
baseline_summary["baseline_process_run_id"] = baseline_process_run_id
baseline_summary["gold_truth_hash"] = GOLD_PARENT_TRUTH_HASH
baseline_summary["gold_truth_path"] = str(GOLD_TRUTH_PATH)
baseline_summary["gold_process_run_id"] = gold_truth.get("process_run_id")
baseline_summary["gold_feature_set_id"] = gold_truth_runtime_facts.get("feature_set_id")

baseline_metadata = {
    "gold_preprocessed_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
    "gold_fit_path": str(GOLD_FIT_DATA_PATH),
    "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
    "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
    "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
    "baseline_models_path": str(BASELINE_MODELS_PATH),
    "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
    "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
    "baseline_metadata_path": str(BASELINE_METADATA_PATH),

    # upstream truth linkage
    "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
    "gold_truth_path": str(GOLD_TRUTH_PATH),
    "gold_process_run_id": gold_truth.get("process_run_id"),
    "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    "gold_scaler_path": gold_truth_artifact_paths.get("scaler_path"),
    "gold_scaler_kind": gold_truth_runtime_facts.get("scaler_kind_runtime"),
    "gold_recommended_imputation": gold_truth_runtime_facts.get("recommended_imputation"),

    # stage truth linkage
    "baseline_truth_hash": BASELINE_TRUTH_HASH,
    "baseline_truth_path": str(baseline_truth_path),
    "baseline_process_run_id": baseline_process_run_id,
}

baseline_results.to_csv(BASELINE_RESULTS_PATH_CSV, index=False)
baseline_results.to_pickle(BASELINE_RESULTS_PATH_PICKLE)


joblib.dump(baseline_model, BASELINE_MODEL_ARTIFACT_PATH)
joblib.dump(baseline_model, BASELINE_MODELS_PATH)

save_json(baseline_thresholds, BASELINE_THRESHOLDS_PATH)
save_json(baseline_summary, BASELINE_SUMMARY_PATH)
save_json(baseline_metadata, BASELINE_METADATA_PATH)

wandb.save(str(BASELINE_RESULTS_PATH_CSV))
wandb.save(str(BASELINE_RESULTS_PATH_PICKLE))
wandb.save(str(BASELINE_MODEL_ARTIFACT_PATH))
wandb.save(str(BASELINE_MODELS_PATH))
wandb.save(str(BASELINE_THRESHOLDS_PATH))
wandb.save(str(BASELINE_SUMMARY_PATH))
wandb.save(str(BASELINE_METADATA_PATH))
wandb.save(str(baseline_truth_path))

ledger.add(
    kind="step",
    step="save_baseline_outputs",
    message="Saved baseline results, trained Isolation Forest model, thresholds, summary, metadata, and baseline stage truth record.",
    data={
        "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
        "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
        "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
        "baseline_models_path": str(BASELINE_MODELS_PATH),
        "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
        "baseline_metadata_path": str(BASELINE_METADATA_PATH),
        "baseline_truth_hash": BASELINE_TRUTH_HASH,
        "baseline_truth_path": str(baseline_truth_path),
        "result_row_count": int(len(baseline_results)),
        "alert_count_all_rows": baseline_alert_count_all_rows,
        "alert_count_test_rows": baseline_alert_count_test_rows,
    },
    logger=logger,
)

2026-06-15 01:00:04,235 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/baseline/thresholds/pump__gold__baseline_thresholds.json
2026-06-15 01:00:04,258 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/baseline/summaries/pump__gold__baseline_summary.json
2026-06-15 01:00:04,283 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/baseline/metadata/pump__gold__baseline_metadata.json
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
2026-06-15 01:00:04,628 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T01:00:04.628513+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'save_baseline_outputs', 'message': 'Saved baseline results, trained Isolation Forest model, thresholds, summary, metadata, and baseline stage truth record.', 'why': None, 'co

{'ts_utc': '2026-06-15T01:00:04.628513+00:00',
 'stage': 'gold_baseline',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'save_baseline_outputs',
 'message': 'Saved baseline results, trained Isolation Forest model, thresholds, summary, metadata, and baseline stage truth record.',
 'why': None,
 'consequence': None,
 'data': {'baseline_results_path_csv': '/workspace/artifacts/gold/pump/baseline/scores/pump__gold__baseline_results.csv',
  'baseline_results_path_pickle': '/workspace/artifacts/gold/pump/baseline/scores/pump__gold__baseline_results.pkl',
  'baseline_model_artifact_path': '/workspace/artifacts/gold/pump/baseline/models/pump__gold__baseline_isolation_forest.joblib',
  'baseline_models_path': '/workspace/models/pump/pump__gold__baseline_isolation_forest.joblib',
  'baseline_thresholds_path': '/workspace/artifacts/gold/pump/baseline/thresholds/pump__gold__baseline_thresholds.json',
  'baseline_summary_path': '/workspace/artifacts/gold/pump/baseline/summari

----

## Build Baseline Detected-Row Review Output

### Ask

Why save detected rows separately from the full baseline results?

### Answer

The full baseline results dataframe contains every row, but the detected-row output gives me a focused review table of only rows that triggered the baseline flag.

That makes later analysis easier because I can inspect baseline alerts without filtering the full results file each time.

In [27]:
baseline_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=baseline_results,
    target_flag_column="baseline_flag",
    row_id_column="meta__row_id",
    score_column="baseline_score",
    decision_column="baseline_decision",
    pred_column="baseline_pred",
    include_columns=[
        "baseline_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_baseline_detected_rows",
    message="Built the baseline detected-rows dataframe from the scored baseline results using stable row tracking.",
    data={
        "target_flag_column": "baseline_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(baseline_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in baseline_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in baseline_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in baseline_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Baseline detected row count: {len(baseline_detected_rows_dataframe):,}")
display(baseline_detected_rows_dataframe.head(20))

2026-06-15 01:00:05,086 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T01:00:05.086156+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'extract_baseline_detected_rows', 'message': 'Built the baseline detected-rows dataframe from the scored baseline results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'baseline_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 87735, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Baseline detected row count: 87,735


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,baseline_flag,baseline_score,baseline_decision,baseline_pred
0,274,16659020197670275987,274,274,2018-04-01 04:34:00+00:00,asset__001,run__001,NORMAL,0,1,0.508947,-0.008947,-1
1,277,2794287318717836113,277,277,2018-04-01 04:37:00+00:00,asset__001,run__001,NORMAL,0,1,0.517735,-0.017735,-1
2,278,6072686403205082512,278,278,2018-04-01 04:38:00+00:00,asset__001,run__001,NORMAL,0,1,0.504978,-0.004978,-1
3,279,9881921393423684931,279,279,2018-04-01 04:39:00+00:00,asset__001,run__001,NORMAL,0,1,0.524405,-0.024405,-1
4,280,17592308112485533406,280,280,2018-04-01 04:40:00+00:00,asset__001,run__001,NORMAL,0,1,0.505015,-0.005015,-1
5,281,10853109780830728877,281,281,2018-04-01 04:41:00+00:00,asset__001,run__001,NORMAL,0,1,0.522094,-0.022094,-1
6,282,13625254895147475258,282,282,2018-04-01 04:42:00+00:00,asset__001,run__001,NORMAL,0,1,0.514290,-0.014290,-1
7,283,5048821507717870176,283,283,2018-04-01 04:43:00+00:00,asset__001,run__001,NORMAL,0,1,0.527602,-0.027602,-1
8,285,14177860280809875926,285,285,2018-04-01 04:45:00+00:00,asset__001,run__001,NORMAL,0,1,0.510907,-0.010907,-1
9,293,10160117446075706286,293,293,2018-04-01 04:53:00+00:00,asset__001,run__001,NORMAL,0,1,0.505839,-0.005839,-1


----

## Finalize the Ledger and Close the Tracking Run

This step writes the baseline ledger to disk and cleanly closes the experiment tracking run.

By the time I get here, the important modeling work and artifact creation are already complete. So this section is mainly about wrapping up the notebook in a structured way.

In [28]:
ledger.add(
    kind="step",
    step="finalize_baseline_modeling",
    message="Gold baseline modeling notebook complete.",
    data={
        "baseline_metrics": baseline_metrics,
        "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
        "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
        "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
        "baseline_model_path": str(BASELINE_MODELS_PATH),
    },
    logger=logger,
)


ledger.write_json(baseline_ledger_path)

wandb.save(str(baseline_ledger_path))
wandb_run.finish()

2026-06-15 01:00:06,117 | INFO | capstone.gold.baseline | LEDGER | {'ts_utc': '2026-06-15T01:00:06.117277+00:00', 'stage': 'gold_baseline', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'finalize_baseline_modeling', 'message': 'Gold baseline modeling notebook complete.', 'why': None, 'consequence': None, 'data': {'baseline_metrics': {'model': 'Baseline IsolationForest', 'threshold_percentile': 97.0, 'threshold': 0.5040984526829705, 'alert_count_all_rows': 87735, 'alert_count_test_rows': 31200, 'precision': 0.003782051282051282, 'recall': 1.0, 'f1': 0.00753560252889712, 'roc_auc': 0.9406852498811833, 'pr_auc': 0.12201014363272505}, 'baseline_results_path_csv': '/workspace/artifacts/gold/pump/baseline/scores/pump__gold__baseline_results.csv', 'baseline_results_path_pickle': '/workspace/artifacts/gold/pump/baseline/scores/pump__gold__baseline_results.pkl', 'baseline_model_artifact_path': '/workspace/artifacts/gold/pump/baseline/models/pump__gold__baseline_isolation_for

----

## Run Final Lineage and Consistency Checks

Before I treat the baseline notebook as complete, I run a final sanity check on the saved baseline results and truth artifacts.

This check verifies things like:
- required lineage columns exist in the results dataframe
- the dataframe truth hash matches the saved baseline truth
- the parent truth hash matches the Gold parent truth
- the saved truth file exists
- the saved metadata points back to the correct baseline truth hash

I like ending with this because it confirms that the baseline output is not only saved, but also internally consistent and traceable.

In [29]:
required_baseline_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_baseline_meta_columns = [
    column_name
    for column_name in required_baseline_meta_columns
    if column_name not in baseline_results.columns
]
if missing_baseline_meta_columns:
    raise ValueError(
        f"baseline_results is missing required lineage columns: {missing_baseline_meta_columns}"
    )

baseline_results_truth_hash_check = extract_truth_hash(baseline_results)
if baseline_results_truth_hash_check is None:
    raise ValueError("baseline_results does not contain a readable meta__truth_hash value.")

if baseline_results_truth_hash_check != BASELINE_TRUTH_HASH:
    raise ValueError(
        "baseline_results truth hash does not match BASELINE_TRUTH_HASH:\n"
        f"dataframe={baseline_results_truth_hash_check}\n"
        f"record={BASELINE_TRUTH_HASH}"
    )

baseline_parent_values = baseline_results["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
if not baseline_parent_values:
    raise ValueError("baseline_results is missing populated meta__parent_truth_hash values.")

if len(baseline_parent_values) != 1:
    raise ValueError(f"baseline_results has multiple parent truth hashes: {baseline_parent_values}")

if baseline_parent_values[0] != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "baseline_results parent truth hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"dataframe_parent={baseline_parent_values[0]}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

if not Path(baseline_truth_path).exists():
    raise FileNotFoundError(f"Baseline truth file was not created: {baseline_truth_path}")

loaded_baseline_truth_raw = load_json(baseline_truth_path)

if not isinstance(loaded_baseline_truth_raw, dict):
    raise TypeError(
        "Saved baseline truth JSON must load as a dictionary. "
        f"Got {type(loaded_baseline_truth_raw).__name__}: {loaded_baseline_truth_raw!r}"
    )

loaded_baseline_truth: dict[str, Any] = loaded_baseline_truth_raw

if loaded_baseline_truth.get("truth_hash") != BASELINE_TRUTH_HASH:
    raise ValueError(
        "Saved Baseline truth file hash does not match BASELINE_TRUTH_HASH:\n"
        f"file={loaded_baseline_truth.get('truth_hash')}\n"
        f"record={BASELINE_TRUTH_HASH}"
    )

if loaded_baseline_truth.get("parent_truth_hash") != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "Saved Baseline truth file parent hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"truth={loaded_baseline_truth.get('parent_truth_hash')}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

saved_baseline_metadata_raw = load_json(BASELINE_METADATA_PATH)

if not isinstance(saved_baseline_metadata_raw, dict):
    raise TypeError(
        "Saved baseline metadata JSON must load as a dictionary. "
        f"Got {type(saved_baseline_metadata_raw).__name__}: {saved_baseline_metadata_raw!r}"
    )

saved_baseline_metadata: dict[str, Any] = saved_baseline_metadata_raw

if saved_baseline_metadata.get("baseline_truth_hash") != BASELINE_TRUTH_HASH:
    raise ValueError(
        "baseline_metadata baseline_truth_hash does not match BASELINE_TRUTH_HASH:\n"
        f"metadata={saved_baseline_metadata.get('baseline_truth_hash')}\n"
        f"record={BASELINE_TRUTH_HASH}"
    )

print("Gold Baseline lineage sanity check passed.")

2026-06-15 01:02:55,838 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/truths/gold_baseline/pump__gold_baseline__truth__037e9f5d72e863d7dadd18c875afe056081c5c4d51cd70bf601b686a10524680.json
2026-06-15 01:02:55,851 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/baseline/metadata/pump__gold__baseline_metadata.json


Gold Baseline lineage sanity check passed.


### Ask

What does this final sanity check tell me?

### Answer

It tells me whether the baseline results can actually be trusted as a pipeline artifact.

A notebook can run successfully and still produce outputs with broken lineage or mismatched truth metadata. This final check helps guard against that by making sure the dataframe, truth record, and saved metadata all agree with each other.

So this is really a trust check, not just a completion check.

----

# Gold Baseline SQL Write Cell
Target:
- gold.anomaly_detection_scores
Purpose:
- Persist baseline Isolation Forest scores and anomaly flags.


In [30]:

WRITE_TO_POSTGRES = True

if WRITE_TO_POSTGRES:
    gold_baseline_sql_summary_dataframe = write_gold_baseline_scores_sql(
        engine=engine,
        capstone_schema=CAPSTONE_SCHEMA,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        notebook_globals=globals(),
        dataset_name=globals().get("DATASET_NAME", DATASET_ID),
    )

    display(gold_baseline_sql_summary_dataframe)
else:
    print("Postgres write skipped.")

        

Using dataframe: baseline_results -> 220,320 rows x 349 columns
Deleted 0 existing rows from gold.anomaly_detection_scores for baseline_isolation_forest/baseline.
Wrote 220,320 rows.
Upserted pipeline run metadata: run__001__gold_baseline_modeling
Logged DQ event: gold.anomaly_detection_scores | baseline_sql_write | pass


,model_name,model_stage,row_count,alert_count
0,baseline_isolation_forest,baseline,220320,87735


## Summary

This notebook preserves the current analytical behavior while documenting the role of the Gold step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

Gold 03 notebooks compare staged cascade modeling against the baseline path.
